# T04: Robustness Lab Notebook

This notebook exercises the robustness surface of the mock KGW watermark:

1. Generate watermarked text with `CreationService` using the test key identifier `test-kgw-v1`.
2. Run a deletion severity sweep with `RobustnessService` at severities `[0.0, 0.1, 0.2, 0.3, 0.4, 0.5]`.
3. Record pre/post scores, decisions, survival flags, and delta scores.
4. Plot a survival curve (severity versus post-attack score and survival flag).

**Warnings**

- All results are produced with the mock provider and a placeholder tokenizer; they are not representative of real language-model behavior.
- Only `key_id` references are stored; no real watermark keys, API keys, or third-party text are included in this notebook.
- `estimated_fpr` is `None` because the calibration corpus is not yet built; this notebook therefore cannot make empirical false-positive claims.
- This notebook attacks only synthetic, self-owned text generated inside the lab. Do not point the transformations at third-party text without authorization.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    """Find the checkout root from any working directory inside the repository."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found; launch from within the checkout.")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from watermark_lab.core.contracts import DetectionConfig
from watermark_lab.run_engine.attack import RobustnessService
from watermark_lab.run_engine.create import CreationService

PROMPT = "The quick brown fox jumps over the lazy dog."
PROFILE_ID = "mock"
METHOD_ID = "kgw"
KEY_ID = "test-kgw-v1"
SEVERITIES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
ATTACK_ID = "deletion"
SEED = "t04"
print("Imports and constants loaded.")

## 1. Generate watermarked text

The generation uses the local mock provider and the same KGW parameters used by the detection step. Governance requires `disclosure_recorded=True`.

In [ ]:
parent = CreationService().create(
    prompt=PROMPT,
    profile_id=PROFILE_ID,
    method_id=METHOD_ID,
    key_id=KEY_ID,
    config={
        "disclosure_recorded": True,
        "max_new_tokens": 80,
        "gamma": 0.25,
        "delta": 2.0,
        "context_width": 1,
    },
)
print(json.dumps({
    "parent_run_id": parent["run_id"],
    "input_sha256": parent["input_sha256"],
    "output_sha256": parent["generation"]["response_sha256"],
    "self_verification": parent["self_verification"],
}, indent=2))

## 2. Configure the keyed detector

The detector uses the same `key_id` and KGW parameters as the generator. This is required for a fair pre/post comparison.

In [ ]:
detector_config = DetectionConfig(
    method_id=METHOD_ID,
    tokenizer_id="gpt2",
    key_id=KEY_ID,
    threshold=4.0,
    options={
        "gamma": 0.25,
        "context_width": 1,
        "seeding_scheme": "simple_1",
    },
)
print(detector_config)

## 3. Run the deletion severity sweep

`RobustnessService.sweep` applies the attack at each severity, detects the original and transformed text, and returns pre/post scores plus a survival flag.

In [ ]:
sweep = RobustnessService().sweep(
    parent, ATTACK_ID, SEVERITIES, detector_config, seed=SEED
)
print(f"Produced {len(sweep)} attack runs")

## 4. Record pre/post scores, decisions, survival flags, and deltas

Each row records the attack severity and the detector outcome before and after the transformation. `survived` is true when the post-attack decision is `verified compatible watermark`.

In [ ]:
records = [
    {
        "severity": r["severity"],
        "pre_score": r["pre_attack"]["score"],
        "pre_decision": r["pre_attack"]["decision"],
        "post_score": r["post_attack"]["score"],
        "post_decision": r["post_attack"]["decision"],
        "survived": r["survived"],
        "delta_score": r["delta_score"],
        "post_token_count": r["post_attack"]["token_count"],
    }
    for r in sweep
]
print(json.dumps(records, indent=2))

## 5. Plot survival curve

The curve shows post-attack z-score versus severity. A horizontal line marks the detection threshold. Runs above the threshold survive the attack.

In [ ]:
import matplotlib.pyplot as plt

severities = [r["severity"] for r in records]
post_scores = [r["post_score"] for r in records]
survived = [r["survived"] for r in records]
threshold = detector_config.threshold

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(severities, post_scores, marker="o", label="post-attack score")
ax.axhline(threshold, color="red", linestyle="--", label=f"threshold={threshold}")
for x, s, v in zip(severities, post_scores, survived):
    ax.annotate("survived" if v else "lost", (x, s), textcoords="offset points", xytext=(0, 10), ha="center")
ax.set_xlabel("deletion severity")
ax.set_ylabel("post-attack z-score")
ax.set_title("KGW watermark survival under deletion attack (mock provider)")
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Assertions and warnings

These checks confirm the sweep produced the expected number of runs, that every pre-attack run verifies, and that the mock-provider limitations are explicit in the output.

In [ ]:
assert len(sweep) == len(SEVERITIES), "Sweep length must match severity list"
assert all(r["pre_attack"]["decision"] == "verified compatible watermark" for r in sweep), "Pre-attack text must verify"
assert all(r["post_attack"]["decision"] in {"verified compatible watermark", "not detected", "inconclusive"} for r in sweep), "Unexpected post-attack decision"
assert parent["model_profile"]["simulated"] is True, "This notebook only runs against the simulated mock provider"
print("All assertions passed. Remember: these results are for the mock provider and placeholder tokenizer only.")

## 7. Save the runtime sweep record

Persist a reduced copy of the sweep in the operating system's temporary directory. Normal notebook execution never writes to committed `project/artifacts`; canonical promotion is a separate, governed workflow. The full text is excluded to keep runtime output small and synthetic-only.

In [ ]:
runtime_dir = (
    Path(tempfile.gettempdir())
    / "llm-watermark-lab"
    / "notebooks"
    / "t04-robustness-lab"
)
runtime_dir.mkdir(parents=True, exist_ok=True)
summary = {
    "parent_run_id": parent["run_id"],
    "input_sha256": parent["input_sha256"],
    "output_sha256": parent["generation"]["response_sha256"],
    "attack_id": ATTACK_ID,
    "seed": SEED,
    "records": records,
}
summary_path = runtime_dir / "notebook-sweep-summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(f"Saved runtime-only summary to {summary_path}")